Через пару дней после вашей инициации в BlackCircuit на внутреннем форуме появилась тревожная новость: легендарный нейроартист Kai Asano устроил цифровой перформанс, который потряс всю сетевую среду. В знак протеста против коммерциализации искусственного творчества, он самолично сжёг датацентр, в котором работал, уничтожив исходный код, скомпилированные модели и даже архив неопубликованных работ.

«Искусство, пережившее бэкап, — уже не искусство. Пусть останется только то, что никто не сможет воспроизвести. Даже я сам»
— Kai Asano, фрагмент манифеста, извлечённый из логов на обгоревшем узле

Однако разведке BlackCircuit удалось частично восстановить один из вычислительных модулей. В нём обнаружены:

Две уцелевшие матрицы KaiNet — авторского трёхступенчатого алгоритма обработки изображений.
Несколько входных изображений в одноканальном формате
Числовые матрицы — результат применения полного алгоритма к этим изображениям (в виде .txt файлов, сохранённых через numpy.savetxt).
Что утрачено:

- Один из трёх фильтров KaiNet — безвозвратно.
- Порядок применения фильтров — неизвестен.

Каждый фильтр KaiNet — это матрица 3×3, применяемая к изображению по скользящему окну (совсем как в незрелых попытках человечества совладать с машинным зрением в далёком 21 веке):
поэлементное произведение окна и фильтра → сумма → в соответствующую ячейку выходной матрицы. Алгоритм последовательно применяет три таких фильтра ко всему изображению.

Ваша задача — восстановить недостающий фильтр и порядок применения всех трёх.

Вам доступен архив:

- algos.csv: две строки по 9 чисел — уцелевшие фильтры.
- Изображения (.png), которые использовались как вход.
- .txt-файлы, которые содержали выход.

Каждому входному FILENAME.png файлу соответствует выходной FILENAME.txt.

От вас ожидается восстановленный алгоритм KaiNet в формате reconstructed_algos.csv, содержащем три строки по 9 чисел через запятую, в порядке применения фильтров. Первая строка соответствует первому фильтру, вторая второму, третья третьему.

Важно:

Точный ответ не требуется: оценивается среднее значение MSE между вашими фильтрами и оригинальными.
У вас есть только 10 попыток. Потом — цифровая тишина.

# INIT

In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from IPython.display import clear_output
import time
import random
import numpy as np

lr_stop = 1e-7
n_epochs_print = 10
default_dtype = torch.float32

# data_dir = "/kaggle/input/hw06-1"
data_dir = "data"

# use GPU if available
device = torch.device(
    "cuda") if torch.cuda.is_available() else torch.device("cpu")

device

device(type='cpu')

In [ ]:
def set_seed(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch (CPU и CUDA)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # если несколько GPU

    # Дополнительные настройки для детерминизма (может снижать производительность)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

# DATASETS

In [ ]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
import matplotlib.pyplot as plt


class PNGDataset(Dataset):

    def __init__(self, data_dir, transform=None):
        """
        Args:
            data_dir (str): Путь к директории с PNG-изображениями.
            transform (callable, optional): Трансформации для изображений.
        """
        self.data_dir = data_dir
        self.transform = transform
        self.png_file_list = [str(i + 1) + ".png" for i in range(0, 1000)]
        # Стандартные трансформации, если не переданы свои

        self.transform = (
            transform
            if transform
            else transforms.Compose(
                [
                    transforms.ToTensor(),  # Конвертация в тензор [0, 1]
                    # Нормализация [-1, 1]
                    transforms.Normalize(mean=[0.5], std=[0.5]),
                ]
            )
        )

    def __len__(self):
        return len(self.png_file_list)

    def __getitem__(self, idx):
        img_name = os.path.join(self.data_dir, self.png_file_list[idx])
        image = Image.open(img_name)
        if self.transform:
            image = self.transform(image)
        return image


class NUMDataset(Dataset):

    def __init__(self, data_dir, transform=None):
        """
        Args:
           data_dir (str): Путь к директории с текстовыми файлами.
        """

        self.data_dir = data_dir
        self.file_list = [str(i + 1) + ".txt" for i in range(0, 1000)]
        self.transform = (
            transform
            if transform
            else transforms.Compose(
                [
                    transforms.ToTensor(),  # Конвертация в тензор [0, 1]
                    transforms.ConvertImageDtype(default_dtype),
                ]
            )
        )

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = os.path.join(self.data_dir, self.file_list[idx])
        num_data = pd.read_csv(file_name, sep=" ", header=None).to_numpy()
        num_data = self.transform(num_data)
        return num_data


class TaskDataset(Dataset):
    def __init__(self, data_dir):
        super().__init__()
        self.png_dataset = PNGDataset(data_dir=data_dir)
        self.num_dataset = NUMDataset(data_dir=data_dir)

    def __len__(self):
        assert len(self.png_dataset) == len(self.num_dataset)
        return len(self.png_dataset)

    def __getitem__(self, idx):
        return self.png_dataset[idx], self.num_dataset[idx]

### CHECK DATASETS

In [4]:
png_data = PNGDataset(data_dir=data_dir)
num_data = NUMDataset(data_dir=data_dir)

assert len(num_data) == len(png_data)

png_names = [f[:-3] for f in png_data.png_file_list]
num_names = [f[:-3] for f in num_data.file_list]

assert png_names == num_names

# MODEL

## NN Functions

In [5]:
from tqdm.auto import tqdm


def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    opt,
    scheduler,
    n_epochs: int,
    device=device,
):
    """
    model: nn tp train
    train_loader, val_loader: data loaders
    loss_fn: loss function to minimize
    opt: optimizer to update NN weights using gradient descent
    n_epochs: number of epochs = number of full data iterations
    """

    train_loss = []
    val_loss = []

    # TRAIN
    for epoch in tqdm(range(n_epochs)):
        ep_train_loss = []
        ep_val_loss = []

        start_time = time.time()
        model.train(True)  # enable dropout / batch_norm training behavior

        trainloss_value = 0

        for X_batch, y_batch in train_loader:
            # move data to target device
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            # train on batch: compute loss, calc grads, perform optimizer step and zero the grads
            preds = model(X_batch)
            loss = loss_fn(preds, y_batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_train_loss.append(loss.item())

        model.train(False)  # disable dropout / use averages for batch_norm

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                # move data to target device

                # YOUR CODE HERE
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                loss = loss_fn(preds, y_batch)
                # compute predictions
                # YOUR CODE HERE
                ep_val_loss.append(loss.item())  # YOUR CODE HERE

        # print the results for this epoch:
        # Print out what's happening
        # print(f"Epoch {epoch + 1} of {n_epochs} took {time.time() - start_time:.3f}s")

        train_loss.append(np.mean(ep_train_loss))

        val_loss.append(np.mean(ep_val_loss))
        scheduler.step(val_loss[-1])

        if not (epoch + 1) % (n_epochs_print):
            print(
                f"Epoch: {epoch+1} | "
                f"train_loss: {train_loss[-1]:.8f} | "
                f"test_loss: {val_loss[-1]:.8f} | "
                f"LR: {opt.param_groups[0]['lr']}"
            )

        if opt.param_groups[0]["lr"] <= lr_stop:
            print("Обучение остановлено: lr слишком мал.")
            break

        # print(f"\t  training loss: {train_loss[-1]:.6f}")
        # print(f"\tvalidation loss: {val_loss[-1]:.6f}")
        # print(f"\tvalidation accuracy: {val_accuracy[-1]:.3f}")

    return train_loss, val_loss

c:\Users\anton\.virtualenvs\03.14_ml-yandex_3.0-9PhsjchO\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import numpy as np
import torch.nn as nn
from itertools import permutations
import copy


def get_permutations_convs():
    conv_list = []
    for i in range(3):
        conv = nn.Conv2d(1, 1, 3, padding=1)
        conv_list.append(conv)

    w1 = [-1.0, -0.5, 0.0, -0.5, 0.5, 0.5, 0.0, 0.5, 1.0]
    w2 = [0.0625, 0.0625, 0.0625, 0.0625,
          0.0625, 0.0625, 0.0625, 0.0625, 0.0625]
    with torch.no_grad():
        w1 = np.array(w1).reshape(3, 3)
        tw1 = torch.Tensor(w1)
        conv_list[0].weight.copy_(tw1)
        conv_list[0].weight.requires_grad = False

        w2 = np.array(w2).reshape(3, 3)
        tw2 = torch.Tensor(w2)
        conv_list[1].weight.copy_(tw2)
        conv_list[1].weight.requires_grad = False

    all_permutations_convs = [copy.deepcopy(
        perm) for perm in permutations(conv_list)]
    return all_permutations_convs

In [7]:
import torch
import torch.nn as nn


class SimpleConvNet(nn.Module):
    def __init__(self):
        super(SimpleConvNet, self).__init__()
        # 1-й слой: 1 канал -> 16 каналов
        self.conv1 = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=3, padding=1)

        # 2-й слой: 16 каналов -> 8 каналов
        self.conv2 = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=3, padding=1)

        # 3-й слой: 8 каналов -> 3 канала (финальные матрицы)
        self.conv3 = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=3, padding=1)

        # ReLU для нелинейности
        self.relu = nn.ReLU()

    def update_convs(self, convs):
        self.conv1 = convs[0]
        self.conv2 = convs[1]
        self.conv3 = convs[2]
        print(f"Convs update is compleated")

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)  # Без ReLU на выходе (если нужны "сырые" значения)
        return x


def output_model(model):
    print(model.conv1.weight)
    print(model.conv2.weight)
    print(model.conv3.weight)

In [8]:
def plot_tensor(image: torch.Tensor):
    image = image.permute(1, 2, 0).squeeze(0).detach().cpu().numpy()
    plt.imshow(image)


def update_tensor(t):
    return t.permute(1, 2, 0).squeeze(0).detach().cpu().numpy()


def plot_pair_tensors(img, data):
    fig, ax = plt.subplots(1, 2)
    img = update_tensor(img)
    data = update_tensor(data)

    ax[0].imshow(img)
    ax[1].imshow(data)

## SPLIT

In [9]:
import numpy as np

np.random.seed(42)


def subset_ind(dataset, ratio: float):
    #     return ### YOUR CODE HERE
    return np.random.choice(len(dataset), size=int(ratio * len(dataset)), replace=False)


dataset = TaskDataset(data_dir=data_dir)

val_size = 0.2
val_inds = subset_ind(dataset, val_size)
train_dataset = Subset(
    dataset, [i for i in range(len(dataset)) if i not in val_inds])
val_dataset = Subset(dataset, val_inds)

assert train_dataset[0][0][0][0][0].item() == 0.3647059202194214

train_dataloader = DataLoader(
    dataset=train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=True)

model = SimpleConvNet()
model.to(device, default_dtype)

loss_func = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss, val_loss = train_model(
    model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    loss_fn=loss_func,
    opt=opt,
    n_epochs=1000,
    device=device,
)

def plot_train_process(train_loss, val_loss):
    fig, axes = plt.subplots(2, 1, figsize=(15, 5))

    axes[0].set_title("Loss")
    axes[0].plot(train_loss, label="train")
    axes[0].plot(val_loss, label="validation")
    axes[0].legend()


plot_train_process(train_loss, val_loss)

print(model.conv1.weight)
print(model.conv2.weight)
print(model.conv3.weight)

model = SimpleConvNet()
model.to(device, default_dtype)

loss_func = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=1e-3)

train_loss, val_loss = train_model(
    model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    loss_fn=loss_func,
    opt=opt,
    n_epochs=1000,
    device=device,
)

plot_train_process(train_loss, val_loss)

print(model.conv1.weight)
print(model.conv2.weight)
print(model.conv3.weight)

In [10]:
import torch.nn as nn
from itertools import permutations
import copy


def get_permutations_convs():

    conv_list = []
    for i in range(3):
        conv = nn.Conv2d(1, 1, 3, padding=1)
        conv_list.append(conv)

    w1 = [-1.0, -0.5, 0.0, -0.5, 0.5, 0.5, 0.0, 0.5, 1.0]
    w2 = [0.0625, 0.0625, 0.0625, 0.0625,
          0.0625, 0.0625, 0.0625, 0.0625, 0.0625]
    w3 = [
        -0.2284,
        -0.2088,
        0.0901,
        0.2287,
        0.3032,
        -0.0377,
        0.1283,
        -0.2647,
        0.1776,
    ]  # some random values
    with torch.no_grad():
        w1 = np.array(w1).reshape(3, 3)
        tw1 = torch.Tensor(w1)
        conv_list[0].weight.copy_(tw1)
        conv_list[0].weight.requires_grad = False

        w2 = np.array(w2).reshape(3, 3)
        tw2 = torch.Tensor(w2)
        conv_list[1].weight.copy_(tw2)
        conv_list[1].weight.requires_grad = False

        w3 = np.array(w3).reshape(3, 3)
        tw3 = torch.Tensor(w3)
        conv_list[2].weight.copy_(tw3)

    all_permutations_convs = [copy.deepcopy(
        perm) for perm in permutations(conv_list)]
    assert not all_permutations_convs[0][0] is conv_list[0]
    return all_permutations_convs


# Проверяем, что это новые объекты
# assert all_permutations_with_copy[0][0] is conv_list[0]  # False (уже копия)


def plot_permutations_loss_results(results):
    for i, res in enumerate(results):
        plt.plot(res[0], label="train_loss_" + str(i))
        plt.plot(res[1], label="test_loss_" + str(i))
        plt.tight_layout()
        plt.legend()


def print_premutation_weights(conv_permutations, idx=-1):
    if idx == -1:
        for i, convs in enumerate(conv_permutations):
            for conv in convs:
                print(conv.weight)
    else:
        for conv in conv_permutations[idx]:
            print(conv.weight)

In [ ]:
conv_permutation = get_permutations_convs()
print_premutation_weights(conv_permutation, 0)

Parameter containing:
tensor([[[[-1.0000, -0.5000,  0.0000],
          [-0.5000,  0.5000,  0.5000],
          [ 0.0000,  0.5000,  1.0000]]]])
Parameter containing:
tensor([[[[0.0625, 0.0625, 0.0625],
          [0.0625, 0.0625, 0.0625],
          [0.0625, 0.0625, 0.0625]]]])
Parameter containing:
tensor([[[[-0.2284, -0.2088,  0.0901],
          [ 0.2287,  0.3032, -0.0377],
          [ 0.1283, -0.2647,  0.1776]]]], requires_grad=True)


: 

In [ ]:
loss_results = []
models = []

for i, convs in enumerate(conv_permutation):
    print("model ", i + 1)
    model = SimpleConvNet()
    model.update_convs(convs=convs)
    model.to(device, default_dtype)

    loss_func = nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=1e-1)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=10, factor=0.1, min_lr=1e-10
    )

    train_loss, val_loss = train_model(
        model,
        train_loader=train_dataloader,
        val_loader=val_dataloader,
        loss_fn=loss_func,
        opt=opt,
        scheduler=scheduler,
        n_epochs=100,
        device=device,
    )

    models.append(model)
    loss_results.append([train_loss, val_loss])

model  1
Convs update is compleated


 10%|█         | 10/100 [00:49<06:25,  4.28s/it]

Epoch: 10 | train_loss: 5197.33111328 | test_loss: 5142.14676339 | LR: 0.1


 12%|█▏        | 12/100 [00:57<06:13,  4.25s/it]

In [ ]:
for i, model in enumerate(models):
    print(i)
    output_model(model)

In [ ]:
plot_permutations_loss_results(loss_results)

In [ ]:
model = SimpleConvNet()
model.update_convs(convs=conv_permutation[-1])
model.to(device, default_dtype)
output_model(model)

In [ ]:
model = SimpleConvNet()
model.update_convs(convs=conv_permutation[-1])
model.to(device, default_dtype)
output_model(model)

In [ ]:
loss_func = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, patience=10, factor=0.1, min_lr=1e-6
)

train_loss, val_loss = train_model(
    model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    loss_fn=loss_func,
    opt=opt,
    scheduler=scheduler,
    n_epochs=500,
    device=device,
)

In [ ]:
plot_permutations_loss_results([[train_loss, val_loss]])

In [ ]:
output_model(model)

# CHECKS and TESTs

In [ ]:
dataset = TaskDataset(data_dir=)
item = dataset[0]

print("len_item=", len(item))
print("type item[0]{}, type_item[1]{}".format(type(item[0]), type(item[1])))

img, data = item
plot_pair_tensors(img, data)

In [ ]:
data_loader = DataLoader(dataset=dataset, batch_size=2)

item = next(iter(data_loader))

In [ ]:
item[0][0].dtype, item[1][0].dtype

### CHECK LOSS FUNCTIONS

In [ ]:
assert 1 == 2

In [ ]:
dataset = TaskDataset(data_dir=data_dir)
data_loader = DataLoader(dataset=dataset, batch_size=16, shuffle=True)

loss_func = nn.MSELoss()
images, labels = next(iter(data_loader))
images = images.to(device)
labels = labels.to(device)

pred = model(images)
loss = loss_func(pred, images)
loss.backward()

## CHECK NUM EPOCHS

In [ ]:
n_epochs = 500
for epoch in range(0, n_epochs):
    if not (epoch + 1) % (20):
        print(epoch + 1)

## CHECK CONVS

In [ ]:
import torch
import random
import numpy as np


def set_seed(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch (CPU и CUDA)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # если несколько GPU

    # Дополнительные настройки для детерминизма (может снижать производительность)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Устанавливаем seed
set_seed(42)

In [ ]:
set_seed(42)
input = torch.randn(1, 1, 64, 64)  # [B, C, H, W]

conv = nn.Conv2d(1, 1, 3, padding=1)
output = conv(input)
print(conv.weight)
print(output.mean())  # [1, 16, 64, 64]

w1 = [-1.0, -0.5, 0.0, -0.5, 0.5, 0.5, 0.0, 0.5, 1.0]
w1 = np.array(w1).reshape(3, 3)
tw1 = torch.Tensor(w1)
tw1
with torch.no_grad():
    conv.weight.copy_(tw1)
output = conv(input)
print(conv.weight)
print(output.mean())  # [1, 16, 64, 64]